In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# ───── Load your file ─────
df = pd.read_excel("Historical Data (2).xlsx", sheet_name="Sheet1")

# ───── Clean data ─────
numeric_cols = ['Monthly Requirement', 'Daily Requirement', 'Buffer Requirement', 'Planned Quantity']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['Daily Requirement', 'Planned Quantity', 'Category',
                       'Monthly Requirement', 'Buffer Requirement', 'Machine', 'Color', 'Shift'])

# Target
df['days_covered'] = df['Planned Quantity'] / df['Daily Requirement']

# ───── Features ─────
cat_cols = ['Machine', 'Color', 'Category', 'Shift']
num_cols = ['Monthly Requirement', 'Buffer Requirement']

# Encode categoricals
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

X = df[cat_cols + num_cols]
y = df['days_covered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ───── Train model ─────
model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
model.fit(X_train, y_train)

# ───── Results ─────
preds = model.predict(X_test)
print("MAE (days error):", round(mean_absolute_error(y_test, preds), 2))
print("R² score:", round(r2_score(y_test, preds), 2))

# Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns)
print("\nTop features:\n", importances.sort_values(ascending=False))

# Save model
model.save_model("days_covered_model.json")
print("\nModel saved as days_covered_model.json")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# ───── Load your file ─────
df = pd.read_excel("Historical Data (2).xlsx", sheet_name="Sheet1")

# ───── Clean data ─────
numeric_cols = ['Monthly Requirement', 'Daily Requirement', 'Buffer Requirement', 
                'Planned Quantity', 'No of Knaban Cards']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows missing critical columns
df = df.dropna(subset=['Daily Requirement', 'Planned Quantity', 'Category',
                       'Monthly Requirement', 'Buffer Requirement', 
                       'Machine', 'Color', 'Shift'])

print("Rows after cleaning:", len(df))

# ───── Target ─────
df['days_covered'] = df['Planned Quantity'] / df['Daily Requirement']

# Clip extreme values (protect from outliers / unrealistic plans)
df['days_covered'] = df['days_covered'].clip(lower=0.1, upper=30)

# Log transform — handles right-skew very well in production planning data
df['target_log'] = np.log1p(df['days_covered'])   # log(1 + x) — safe if any near-zero

# ───── Features ─────
cat_cols = ['Machine', 'Color', 'Category', 'Shift']
num_cols = ['Monthly Requirement', 'Buffer Requirement']

# Encode categoricals
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

X = df[cat_cols + num_cols]
y = df['target_log']

# ───── Split ─────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

# Show original-scale stats (after clip) for reference
print("\ndays_covered (original, clipped) stats:")
print(np.expm1(y).describe())

# ───── Model ─────
model = xgb.XGBRegressor(
    n_estimators=80,
    learning_rate=0.08,
    max_depth=3,              # shallower tree → better on small data
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.0,           # light L2 regularization
    random_state=42,
    objective='reg:squarederror'
)

model.fit(X_train, y_train)

# ───── Predict & evaluate on ORIGINAL scale ─────
preds_log = model.predict(X_test)
preds_orig = np.expm1(preds_log)          # back to original days scale

y_test_orig = np.expm1(y_test)

mae = mean_absolute_error(y_test_orig, preds_orig)
r2  = r2_score(y_test_orig, preds_orig)

print("\nMAE (days error):", round(mae, 2))
print("R² score:", round(r2, 2))

# ───── Feature importance ─────
importances = pd.Series(model.feature_importances_, index=X.columns)
print("\nFeature importances:")
print(importances.sort_values(ascending=False))

# ───── Save model ─────
model.save_model("days_covered_model_improved.json")
print("\nModel saved as days_covered_model_improved.json")

In [ ]:
import pandas as pd
import numpy as np

# =============================================================
#   SETTINGS - change these according to your file
# =============================================================

FILE_PATH = "Historical Data (2).xlsx"   # ← your current file
SHEET_NAME = "Sheet1"

# Thresholds - adjust based on your experience
SHORTFALL_WARNING_DAYS = 3      # if shortfall > 3 days → urgent warning
SAFE_SURPLUS_DAYS = 2           # if surplus >= 2 days → probably safe

# =============================================================

def load_stranger_data(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    
    # make sure numeric columns are clean
    for col in ['Planned Quantity', 'Actual Production', 'Daily Requirement']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # keep only Stranger category
    df = df[df['Category'].astype(str).str.contains('Stranger', case=False, na=False)].copy()
    
    df = df.dropna(subset=['Part', 'Planned Quantity', 'Actual Production', 'Daily Requirement'])
    
    return df


def calculate_coverage(df):
    df = df.copy()
    
    # days calculations
    df['planned_days']    = df['Planned Quantity'] / df['Daily Requirement']
    df['produced_days']   = df['Actual Production'] / df['Daily Requirement']
    df['difference_days'] = df['produced_days'] - df['planned_days']
    
    # simple interpretation
    df['status'] = df['difference_days'].apply(
        lambda x: 'Shortfall - plan again sooner' if x < -SHORTFALL_WARNING_DAYS else
                  'Slight shortfall - monitor' if x < 0 else
                  'On target or better' if x <= SAFE_SURPLUS_DAYS else
                  'Good surplus - safe for longer'
    )
    
    # round for readability
    for col in ['planned_days', 'produced_days', 'difference_days']:
        df[col] = df[col].round(1)
    
    # select useful columns
    result = df[[
        'Date', 'Part', 'Color', 'Machine', 'Shift',
        'Planned Quantity', 'Actual Production',
        'Daily Requirement',
        'planned_days', 'produced_days', 'difference_days',
        'status'
    ]].sort_values(['Date', 'Part'])
    
    return result


# ─── Run ───
df_stranger = load_stranger_data(FILE_PATH, SHEET_NAME)
summary = calculate_coverage(df_stranger)

print("\nStranger Parts - Coverage Analysis\n")
print(summary.to_string(index=False))

# Optional: save to Excel/CSV
# summary.to_excel("stranger_coverage_report.xlsx", index=False)
# summary.to_csv("stranger_coverage_report.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

# =============================================================
# SETTINGS - change these according to your file
# =============================================================
FILE_PATH = "Historical Data (2).xlsx"   # ← your current file
SHEET_NAME = "Sheet1"

# Thresholds - adjust based on your experience
SHORTFALL_WARNING_DAYS = 3      # if shortfall > 3 days → urgent warning
SAFE_SURPLUS_DAYS = 2           # if surplus >= 2 days → probably safe

# =============================================================

def load_stranger_data(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    
    # make sure numeric columns are clean
    numeric_cols = [
        'Planned Quantity', 'Actual Production', 'Daily Requirement',
        'Monthly Requirement', 'Buffer Requirement'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # keep only Stranger category
    df = df[df['Category'].astype(str).str.contains('Stranger', case=False, na=False)].copy()
    
    # drop rows missing key columns
    df = df.dropna(subset=['Part', 'Planned Quantity', 'Actual Production', 'Daily Requirement'])
    
    return df


def calculate_coverage(df):
    df = df.copy()
    
    # days calculations
    df['planned_days']    = df['Planned Quantity'] / df['Daily Requirement']
    df['produced_days']   = df['Actual Production'] / df['Daily Requirement']
    df['difference_days'] = df['produced_days'] - df['planned_days']
    
    # simple interpretation
    df['status'] = df['difference_days'].apply(
        lambda x: 'Shortfall - plan again sooner' if x < -SHORTFALL_WARNING_DAYS else
                  'Slight shortfall - monitor' if x < 0 else
                  'On target or better' if x <= SAFE_SURPLUS_DAYS else
                  'Good surplus - safe for longer'
    )
    
    # round for readability
    for col in ['planned_days', 'produced_days', 'difference_days',
                'Monthly Requirement', 'Daily Requirement']:
        if col in df.columns:
            df[col] = df[col].round(1)
    
    # select useful columns (now including monthly & daily)
    columns_to_show = [
        'Date', 'Part', 'Color', 'Machine', 'Shift',
        'Monthly Requirement', 'Daily Requirement',
        'Planned Quantity', 'Actual Production',
        'planned_days', 'produced_days', 'difference_days',
        'status'
    ]
    # only include columns that actually exist
    columns_to_show = [c for c in columns_to_show if c in df.columns]
    
    result = df[columns_to_show].sort_values(['Date', 'Part'])
    
    return result


# ─── Run ───
df_stranger = load_stranger_data(FILE_PATH, SHEET_NAME)
summary = calculate_coverage(df_stranger)

print("\nStranger Parts - Coverage Analysis\n")
print(summary.to_string(index=False))

# Summary statistics for Stranger parts
if not summary.empty:
    print("\n" + "="*60)
    print("Summary for Stranger parts:")
    print(f"Number of Stranger runs: {len(summary)}")
    print(f"Average Monthly Requirement: {summary['Monthly Requirement'].mean():.0f}")
    print(f"Average Daily Requirement:   {summary['Daily Requirement'].mean():.1f}")
    print(f"Median Monthly Requirement:  {summary['Monthly Requirement'].median():.0f}")
    print(f"Median Daily Requirement:    {summary['Daily Requirement'].median():.1f}")
    print("="*60 + "\n")
else:
    print("\nNo Stranger parts found in the data.")